# Showing things that are not text

A cell can print. This notebook is about the other channel: `display`,
and the three things built on it — **plots**, **event streams** and
**controls**.

Everything here runs on the kernel in the page. Nothing is fetched, and
nothing about the picture comes from the kernel: Lua sends numbers, the
Lab draws them. That is what lets a chart follow the page's theme, resize
with the window, and be stored in the `.ipynb` as data rather than markup.

## Plots

`plot.line` takes a list of `y` values, or `x` and `y`.

In [ ]:
local ys = {}
for i = 1, 40 do ys[i] = math.sin(i / 5) * (1 - i / 60) end
plot.line(ys, nil, { title = "A decaying sine", x_label = "step" })

Several series, each named. The legend is always there for two or more,
and the colours are assigned in a fixed order — so removing a series
never repaints the ones that are left.

There is deliberately no second y axis. Two measures on different
scales get two charts, or get indexed to a common base; a dual axis
lets whoever drew it choose where the lines cross.


In [ ]:
local sample, smoothed = {}, {}
local acc = 0
for i = 1, 60 do
  sample[i] = math.sin(i / 7) + (i % 5) / 6 - 0.4
  acc = acc + (sample[i] - acc) * 0.2
  smoothed[i] = acc
end
plot{
  title = "A noisy signal and its running mean",
  x_label = "sample", y_label = "value",
  series = {
    { name = "raw", y = sample },
    { name = "smoothed", y = smoothed },
  },
}

`plot.bar` takes labels and values. A bar chart's axis always includes
zero — cropping it would misstate every ratio it draws.

In [ ]:
plot.bar(
  { "parse", "compile", "dump", "load", "run" },
  { 12, 31, 4, 6, 88 },
  { title = "Time by phase (ms)" })

`plot.scatter` for points with no order between them. A value that is
not a number — `nil`, NaN, an infinity — is drawn as a **gap**, never as
a zero, because a chart that fills in a missing measurement is a chart
that publishes something nobody measured.

In [ ]:
local xs, ys = {}, {}
for i = 1, 30 do
  xs[i] = i + math.sin(i) * 2
  ys[i] = i * 0.7 + math.cos(i * 3) * 3
end
plot.scatter(xs, ys, { title = "Thirty points", x_label = "x", y_label = "y" })

-- and a gap, which is a hole in the line rather than a dive to zero
plot.line({ 1, 2, 0/0, 4, 5 }, nil, { title = "A missing measurement" })

Every chart has a **Table** button. It is not a fallback: some of the
series colours are deliberately low-contrast against a white page, and
the rule for that is that the numbers have to be reachable another way.

## Event streams

`events` renders a list of records in the shape Diluvium's swarm layer
emits on `system/events` — `event`, `id`, `detail`. The eight kinds it
knows are the eight `doc/Messaging.md` §9.2 lists.

**The records below are written by hand.** The Lab cannot yet drive a
real swarm: `dvs.c` is not in the WASM artifact, so there is nothing in
the browser to drain. What is real is the renderer and the schema — see
ROADMAP §"A swarm runner" for the one change to the artifact that would
connect them.

In [ ]:
events({
  { event = "spawned",  id = 2 },
  { event = "spawned",  id = 3 },
  { event = "denied",   id = 3, detail = "capability not held: lifecycle" },
  { event = "throttled",id = 1, detail = "spawn rate limit: 4 per step" },
  { event = "exceeded", id = 2, detail = "instruction budget: 200000" },
  { event = "faulted",  id = 3, detail = "attempt to index a nil value (field 'inbox')" },
  { event = "exited",   id = 2 },
}, { title = "one step of a swarm" })

## Controls

A control's callback is an ordinary Lua closure, and it stays in the
kernel — it captured locals that exist nowhere else. The page holds an
id and asks for the function by name when the control moves.

So the callback can see everything the cell could:

In [ ]:
local base = 2

widget.slider{ label = "terms", min = 2, max = 24, value = 8,
  on_change = function(n)
    local ys = {}
    for i = 1, n do ys[i] = base ^ i end
    plot.line(ys, nil, { title = n .. " powers of " .. base })
  end }

`widget.select`, `widget.checkbox` and `widget.button` work the same way.

In [ ]:
widget.select{ label = "shape", options = { "sin", "cos", "square" }, value = "sin",
  on_change = function(which)
    local f = which == "sin" and math.sin
      or which == "cos" and math.cos
      or function(t) return math.sin(t) > 0 and 1 or -1 end
    local ys = {}
    for i = 1, 60 do ys[i] = f(i / 6) end
    plot.line(ys, nil, { title = which })
  end }

A control's output lands under the control, and is **not** saved with
the notebook. What gets saved is the control and the last value it had:
a file should carry the result the cell produced, not the last frame of
someone dragging a slider.

Reopen this notebook without running it and the controls are there but
disabled — the kernel that held their callbacks is gone, and a slider
that moves and does nothing would be a worse lie than one that says so.

## The primitive underneath

All of the above is `display`, which takes a mime bundle exactly as
Jupyter's `display_data` does. The richest representation the page
understands wins; `text/plain` always ships alongside, so a notebook
written here still says something in a reader that has never heard of
Diluvium.

In [ ]:
display{
  ["text/plain"] = "a red circle",
  ["image/svg+xml"] = [[<svg xmlns="http://www.w3.org/2000/svg" width="60" height="60">
    <circle cx="30" cy="30" r="24" fill="#e34948"/></svg>]],
}

SVG is the one type that carries markup, so it is filtered on the way
in: shapes and text survive, and anything that can run or fetch does
not. A notebook is untrusted input — it arrives from files and other
people's repositories — and opening one must never be able to run
anything.